# 📊 Kaggle Playground Series S6E11: Comprehensive EDA
### Predicting Credit Card Fraud | Nhóm Học Máy (Member A: Trần Nhật Hưng)

- **Competition:** [Predicting Credit Card Fraud - Playground Series S6E11](https://www.kaggle.com/competitions/playground-series-s6e11)
- **Evaluation Metric:** **ROC AUC** (dự đoán xác suất `is_fraud` trong khoảng $[0, 1]$)
- **Mục tiêu giai đoạn 1:** Khám phá cấu trúc dữ liệu, phân tích mất cân bằng mẫu, kiểm tra tính đơn điệu của các đặc trưng rủi ro và định hướng kỹ thuật tạo đặc trưng (Feature Engineering) cho Giai đoạn 2.


In [ ]:
import os
import sys
import gc
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score

# Hiển thị cấu hình
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 1000)
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")


## 1. Thiết lập Môi trường & Nạp Dữ liệu
Notebook tự động nhận diện chạy trên **máy Local** hoặc trên **Kaggle Kernel**.


In [ ]:
# Tự động nhận diện đường dẫn
KAGGLE_PATH = Path("/kaggle/input/playground-series-s6e11")
LOCAL_PATH = Path("data/raw")

if KAGGLE_PATH.exists():
    DATA_DIR = KAGGLE_PATH
    print(f"==> Đang chạy trên KAGGLE: {DATA_DIR}")
elif LOCAL_PATH.exists():
    DATA_DIR = LOCAL_PATH
    print(f"==> Đang chạy trên LOCAL: {DATA_DIR}")
else:
    DATA_DIR = Path("../data/raw")
    print(f"==> Đang chạy trên LOCAL (rel): {DATA_DIR}")

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
sub_sample = pd.read_csv(DATA_DIR / "sample_submission.csv")

print(f"Tập Train shape: {train_df.shape}")
print(f"Tập Test shape : {test_df.shape}")
print(f"Sample Sub shape: {sub_sample.shape}")


## 2. Kiểm tra Chất lượng Dữ liệu (Data Hygiene)
Kiểm tra giá trị thiếu (missing values), kiểu dữ liệu và tính tương thích giữa tập train và test.


In [ ]:
# Kiểm tra Missing values & Dtypes
train_nulls = train_df.isnull().sum()
test_nulls = test_df.isnull().sum()

print("Số lượng missing values trong tập Train:", (train_nulls > 0).sum())
print("Số lượng missing values trong tập Test :", (test_nulls > 0).sum())

# Danh sách cột
numeric_cols = [c for c in train_df.select_dtypes(include=[np.number]).columns if c not in ["id", "is_fraud"]]
bool_cols = train_df.select_dtypes(include=[bool]).columns.tolist()
cat_cols = train_df.select_dtypes(include=["object"]).columns.tolist()

print(f"\n* Numerical features ({len(numeric_cols)}): {numeric_cols}")
print(f"* Boolean risk flags ({len(bool_cols)}): {bool_cols}")
print(f"* Categorical features ({len(cat_cols)}): {cat_cols}")


## 3. Phân tích Biến Mục Tiêu (`is_fraud`) & Mất Cân Bằng Mẫu
Bài toán phân loại gian lận thẻ tín dụng đặc trưng bởi **mất cân bằng mẫu cực cao** (Class Imbalance).
Metric đánh giá là **ROC AUC**, do đó mô hình cần tối ưu hóa thứ hạng xác suất dự đoán thay vì ngưỡng cứng 0.5.


In [ ]:
target_counts = train_df["is_fraud"].value_counts()
target_pct = train_df["is_fraud"].value_counts(normalize=True) * 100

print(f"Hợp lệ (0): {target_counts[0]:,} mẫu ({target_pct[0]:.2f}%)")
print(f"Gian lận (1): {target_counts[1]:,} mẫu ({target_pct[1]:.2f}%)")
print(f"Tỷ lệ mất cân bằng: 1 gian lận trên mỗi {int(target_counts[0]/target_counts[1])} giao dịch hợp lệ.")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
colors = ["#2ecc71", "#e74c3c"]
ax1.bar(["Hợp lệ (0)", "Gian lận (1)"], target_counts.values, color=colors, width=0.45)
for i, v in enumerate(target_counts.values):
    ax1.text(i, v + 12000, f"{v:,}\n({v/len(train_df):.2%})", ha="center", fontweight="bold")
ax1.set_title("Phân bố số lượng nhãn target", fontsize=12, fontweight="bold")
ax1.set_ylabel("Số lượng mẫu")
ax1.set_ylim(0, max(target_counts.values) * 1.15)

ax2.pie(target_counts.values, labels=["Hợp lệ", "Gian lận"], autopct="%1.2f%%", colors=colors, explode=(0, 0.15), startangle=90)
ax2.set_title("Tỷ lệ % biến mục tiêu", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


## 4. Phân tích Các Cờ Rủi Ro Boolean (Risk Flags & Fraud Lift)
Bộ dữ liệu cung cấp 6 cờ boolean thể hiện các dấu hiệu rủi ro cao:
- `is_foreign_transaction`: Giao dịch ở nước ngoài
- `ip_country_mismatch`: Quốc gia IP không khớp với thẻ
- `billing_shipping_mismatch`: Địa chỉ thanh toán khác địa chỉ giao hàng
- `is_ai_generated_scam_attempt`: Dấu hiệu lừa đảo tự động hóa / AI
- `used_vpn`: Sử dụng mạng riêng ảo VPN
- `is_new_merchant`: Giao dịch lần đầu với đơn vị bán hàng này


In [ ]:
risk_stats = []
for c in bool_cols:
    t_rate = train_df[train_df[c] == True]["is_fraud"].mean() * 100
    f_rate = train_df[train_df[c] == False]["is_fraud"].mean() * 100
    lift = t_rate / (f_rate if f_rate > 0 else 1e-5)
    t_cnt = train_df[c].sum()
    risk_stats.append({
        "Feature": c,
        "True Count": f"{t_cnt:,}",
        "Fraud Rate when True (%)": f"{t_rate:.2f}%",
        "Fraud Rate when False (%)": f"{f_rate:.2f}%",
        "Lift (Lần)": f"{lift:.1f}x",
        "lift_val": lift
    })

risk_df = pd.DataFrame(risk_stats).sort_values("lift_val", ascending=False)
display(risk_df.drop(columns=["lift_val"]))

# Vẽ biểu đồ Lift
fig, ax = plt.subplots(figsize=(10, 5))
plot_df = pd.DataFrame(risk_stats).sort_values("lift_val", ascending=True)
bars = ax.barh(plot_df["Feature"], plot_df["lift_val"], color="#e74c3c")
for b in bars:
    ax.text(b.get_width() + 0.15, b.get_y() + b.get_height()/2, f"{b.get_width():.1f}x", va="center", fontweight="bold")
ax.set_title("Hệ số rủi ro (Fraud Lift) của từng cờ Boolean", fontsize=12, fontweight="bold")
ax.set_xlabel("Tỷ lệ tăng rủi ro (Lift Multiplier)")
ax.set_xlim(0, max(plot_df["lift_val"]) * 1.2)
plt.tight_layout()
plt.show()


## 5. Phân tích Các Biến Định Tính (Categorical Features)
Khám phá tỷ lệ gian lận theo `merchant_category`, `auth_method`, `channel`, `card_type`, `device_type`.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 11))

# 1. Merchant Category
cat_summary = train_df.groupby("merchant_category")["is_fraud"].agg(["count", "mean"]).sort_values("mean", ascending=True)
axes[0, 0].barh(cat_summary.index, cat_summary["mean"] * 100, color="#3498db")
axes[0, 0].axvline(train_df["is_fraud"].mean() * 100, color="red", linestyle="--", label="Tỷ lệ TB")
axes[0, 0].set_title("Tỷ lệ Gian lận theo Ngành hàng (%)", fontweight="bold")
axes[0, 0].set_xlabel("Tỷ lệ gian lận (%)")
axes[0, 0].legend()

# 2. Auth Method
auth_summary = train_df.groupby("auth_method")["is_fraud"].agg(["count", "mean"]).sort_values("mean", ascending=True)
axes[0, 1].barh(auth_summary.index, auth_summary["mean"] * 100, color="#e67e22")
axes[0, 1].axvline(train_df["is_fraud"].mean() * 100, color="red", linestyle="--")
axes[0, 1].set_title("Tỷ lệ Gian lận theo Phương thức Xác thực (%)", fontweight="bold")
axes[0, 1].set_xlabel("Tỷ lệ gian lận (%)")

# 3. Channel
ch_summary = train_df.groupby("channel")["is_fraud"].agg(["count", "mean"]).sort_values("mean", ascending=True)
axes[1, 0].bar(ch_summary.index, ch_summary["mean"] * 100, color="#2ecc71", width=0.5)
axes[1, 0].axhline(train_df["is_fraud"].mean() * 100, color="red", linestyle="--")
axes[1, 0].set_title("Tỷ lệ Gian lận theo Kênh giao dịch (%)", fontweight="bold")
axes[1, 0].set_ylabel("Tỷ lệ gian lận (%)")

# 4. Card Type
card_summary = train_df.groupby("card_type")["is_fraud"].agg(["count", "mean"]).sort_values("mean", ascending=True)
axes[1, 1].bar(card_summary.index, card_summary["mean"] * 100, color="#9b59b6", width=0.5)
axes[1, 1].axhline(train_df["is_fraud"].mean() * 100, color="red", linestyle="--")
axes[1, 1].set_title("Tỷ lệ Gian lận theo Loại thẻ (%)", fontweight="bold")
axes[1, 1].set_ylabel("Tỷ lệ gian lận (%)")

plt.tight_layout()
plt.show()


## 6. Phân tích Biến Định Lượng (Numerical Features) & Phân Phối
Đặc biệt chú ý các thuộc tính có tương quan mạnh nhất: `cvv_retry_count`, `merchant_risk_score`, `velocity_score`, `amount_usd`.


In [ ]:
# Bảng thống kê mô tả
num_desc = train_df[numeric_cols].describe().T[["mean", "std", "min", "25%", "50%", "75%", "max"]]
display(num_desc)

# Phân tích CVV Retry Count
cvv_analysis = train_df.groupby("cvv_retry_count")["is_fraud"].agg(["count", "mean"])
cvv_analysis["fraud_rate_pct"] = cvv_analysis["mean"] * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.bar(cvv_analysis.index, cvv_analysis["fraud_rate_pct"], color="#c0392b", width=0.45)
for i, r in cvv_analysis.iterrows():
    ax1.text(i, r["fraud_rate_pct"] + 1, f"{r['fraud_rate_pct']:.1f}%\n(N={int(r['count']):,})", ha="center", fontweight="bold")
ax1.set_title("Tỷ lệ Gian lận tăng phi mã theo số lần thử sai CVV", fontsize=11, fontweight="bold")
ax1.set_xlabel("Số lần nhập lại CVV (cvv_retry_count)")
ax1.set_ylabel("Tỷ lệ gian lận (%)")
ax1.set_ylim(0, max(cvv_analysis["fraud_rate_pct"]) * 1.25)

# Merchant Risk Score KDE
sns.kdeplot(data=train_df[train_df["is_fraud"]==0]["merchant_risk_score"], label="Hợp lệ (0)", ax=ax2, color="#2ecc71", fill=True, alpha=0.3)
sns.kdeplot(data=train_df[train_df["is_fraud"]==1]["merchant_risk_score"], label="Gian lận (1)", ax=ax2, color="#e74c3c", fill=True, alpha=0.3)
ax2.set_title("Phân phối Điểm Rủi ro Đơn vị bán lẻ (Merchant Risk Score)", fontsize=11, fontweight="bold")
ax2.set_xlabel("Điểm rủi ro")
ax2.legend()
plt.tight_layout()
plt.show()


## 7. Phân tích Ma Trận Tương Quan (Correlation Matrix)
Đánh giá độ tương quan tuyến tính giữa các biến số với mục tiêu `is_fraud`.


In [ ]:
all_features_corr = numeric_cols + bool_cols + ["is_fraud"]
corr = train_df[all_features_corr].astype(float).corr()["is_fraud"].sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 8))
corr.drop("is_fraud").plot(kind="barh", ax=ax, color=np.where(corr.drop("is_fraud") > 0, "#e74c3c", "#3498db"))
ax.set_title("Hệ số tương quan (Pearson) của các biến với is_fraud", fontsize=12, fontweight="bold")
ax.set_xlabel("Correlation coefficient")
plt.tight_layout()
plt.show()


## 8. Kết luận EDA & Định Hướng Feature Engineering (Giai đoạn 2)

Từ kết quả khám phá dữ liệu, nhóm rút ra các kết luận trọng tâm:

1. **Tín hiệu rủi ro cực mạnh (High-impact Signals):**
   - **`cvv_retry_count`**: Khi thử sai CVV $\ge 2$ lần, tỷ lệ gian lận tăng đột biến (tương quan hàng đầu $r \approx 0.21$).
   - **Các cờ Boolean**: `ip_country_mismatch` (Lift 11.6x), `billing_shipping_mismatch` (Lift 8.5x), `is_ai_generated_scam_attempt` (Lift 9.4x), `is_foreign_transaction` (Lift 8.5x).
   - **`merchant_risk_score` & `velocity_score`**: Phân phối phân tách rõ rệt giữa hai nhóm nhãn.
   - **`auth_method`**: Các giao dịch `No Authentication` có nguy cơ gian lận cao gấp hơn 10 lần so với `Biometric` hoặc `PIN`.
   - **Ngành hàng**: `Gift Cards`, `Crypto Exchange`, `Gaming` có rủi ro cao nhất ($5 - 7.5\%$).

2. **Định hướng Feature Engineering cho Giai đoạn 2:**
   - **Risk Score tổng hợp:** Đếm số cờ rủi ro (`risk_flags_count = is_foreign + ip_mismatch + billing_mismatch + used_vpn + ...`).
   - **Tương tác đặc trưng (Interactions):** 
     - Tỷ lệ `amount_usd / account_balance_usd` (tỷ trọng số dư bị rút trong giao dịch).
     - Rủi ro kết hợp: `merchant_risk_score * velocity_score`.
     - Tốc độ giao dịch: `amount_usd / (hours_since_last_txn + 1)`.
   - **Mã hóa mục tiêu (Target Encoding) có kiểm soát Out-of-Fold:** áp dụng cho `merchant_category`, `auth_method` để khai thác triệt để phân tầng rủi ro.
   - **Tần suất giao dịch:** Tần suất theo `device_type`, `channel` theo từng khung giờ trong ngày.
